# CEHR-GPT COPD cohort 

**This is a multi-session run.** Sequence build ~hours (Spark), prep ~1h, training ~3.3h/epoch. Bank after each stage; after a wipe re-run §0 + §1 and resume.

## §Config paths, aliases, run switches
One place for every path and Foundry alias. Run first, and after every wipe.

In [ ]:
import os, sys

REPO = "/home/user/repo"
INCLUDE_DEATH = True          # stage death so DeathEventDecorator adds death events
INCLUDE_VALUES = True

PKG_CEHRBERT      = f"{REPO}/cehrbert-main"
PKG_CEHRGPT       = f"{REPO}/cehrgpt-main/src"
PKG_CEHRBERT_DATA = f"{REPO}/cehrbert_data-main/src"
for _p in (PKG_CEHRBERT, PKG_CEHRGPT, PKG_CEHRBERT_DATA):
    if _p not in sys.path: sys.path.insert(0, _p)

GPT_UTILS_PY = f"{PKG_CEHRGPT}/cehrgpt/gpt_utils.py"
TOKENIZER_PY = f"{PKG_CEHRGPT}/cehrgpt/models/tokenization_hf_cehrgpt.py"
APP_QCL      = f"{PKG_CEHRBERT_DATA}/cehrbert_data/apps/generate_included_concept_list.py"
APP_SEQ      = f"{PKG_CEHRBERT_DATA}/cehrbert_data/apps/generate_training_data.py"

OMOP_DIR     = f"{REPO}/omop_synthea"
DATA_DIR     = f"{REPO}/cehrgpt_output_full"
PATIENT_SEQ  = f"{DATA_DIR}/patient_sequence"
DATASET_PREP = f"{DATA_DIR}/dataset_prepared_real"
MODEL_DIR    = f"{REPO}/model_run_full"          # tokenizer in / fresh build
MODEL_OUT    = f"{REPO}/model_run_full_real"     # trained model out
SYNTH_OUT    = f"{REPO}/synthetic_full_v3"
LOG_DIR      = f"{REPO}/logs"
SPARK_TMP    = f"{REPO}/spark_tmp"
LOG4J        = f"{REPO}/log4j2.properties"
MAX_SEQ_LEN  = 1024                              # length filter (values make sequences heavier)

# Foundry aliases create them in the UI, then restart the kernel
ALIAS_SEQ_V3   = "copd_patient_sequence_v3"
ALIAS_MODEL_V3 = "copd_cehrgpt_v3_resume"
ALIAS_PREP_V3  = "copd_dataset_prepared_v3"

os.environ.update(OMOP_DIR=OMOP_DIR, CEHR_GPT_DATA_DIR=DATA_DIR, CEHR_GPT_MODEL_DIR=MODEL_DIR)
for d in (DATA_DIR, LOG_DIR, SPARK_TMP): os.makedirs(d, exist_ok=True)
print(f"REPO={REPO} | death={INCLUDE_DEATH} | values={INCLUDE_VALUES} | max_seq={MAX_SEQ_LEN}")

## §0 Environment (after every wipe)
Maestro rebuild, then layer the CUDA torch build. Restart the kernel when prompted and rerun until all OK.

In [ ]:
"""
CELL 1 - ENVIRONMENT DIAGNOSTIC + AUTO-INSTALL via Maestro's own CLI.
Does NOT edit meta.yaml or lock files. For anything missing or mis-versioned it runs
the same command the GUI runs:   maestro env conda install "<spec>"
Set AUTO_INSTALL = False to make this cell report-only.
After an install round: RESTART THE KERNEL, then re-run this cell until all-OK.

Reference target env (also what this cell enforces):
  conda: openjdk==17.0.18 | pytorch>=2.4 | numpy>=1.24,<2 | pandas>=2.0,<2.2 (NOT 2.2.0)
  rest:  pyspark==3.5.5 | pyarrow | pyarrow-hotfix | datasets==2.16.1 | transformers==4.44.1
         tokenizers==0.19.0 | accelerate>=0.31.0 | peft==0.10.0 | optuna==4.0.0
         evaluate==0.4.1 | scikit-learn==1.4.0 | scipy==1.12.0 | xgboost==2.0.3
         dask==2024.1.1 | meds==0.3.3 | meds_reader==0.1.9 | femr>=0.2.0,<=0.2.4
         fast-ml==3.68 | openai==1.54.3 | lightgbm | polars | wandb>=0.17.8
         jinja2==3.1.3 | networkx>=3.2.1 | pillow>=10.3.0 | python-dateutil==2.8.2
         pyyaml==6.0.1 | tqdm>=4.66.1 | werkzeug==3.0.1 | setuptools<80
NEVER install: cehrbert, cehrbert_data, cehrgpt (local folders; a pip copy shadows them).
NEVER put meds/meds_reader/femr/fast-ml in the GUI pip section - their declared deps
conflict with the pins (meds_reader wants pandas>=2.2); this cell layers them --no-deps.
Python itself: pins were validated on 3.11 - this cell will NOT change python;
if the solver fights you on 3.12, pin python==3.11 in the GUI.
"""
import os, sys, subprocess, importlib, importlib.util, importlib.machinery
import importlib.metadata as md
from pathlib import Path

AUTO_INSTALL = True          # set False to diagnose only

OK, BAD = [], []
def check(name, good, detail=""):
    (OK if good else BAD).append((name, detail))
    print(f"  {'OK  ' if good else 'FIX '} {name:30s} {detail}")

# ---------- runtime env vars (process-level only) ----------
JVM_OPTS = " ".join(f"--add-opens=java.base/{m}=ALL-UNNAMED" for m in [
    "java.lang","java.lang.invoke","java.lang.reflect","java.io","java.net","java.nio",
    "java.util","java.util.concurrent","sun.nio.ch","sun.nio.cs","sun.security.action","sun.util.calendar"])
os.environ["JAVA_TOOL_OPTIONS"] = os.environ["_JAVA_OPTIONS"] = JVM_OPTS
os.environ["JAVA_HOME"] = sys.prefix
if f"{sys.prefix}/bin" not in os.environ["PATH"]:
    os.environ["PATH"] = f"{sys.prefix}/bin:" + os.environ["PATH"]

print(f"  python {sys.version.split()[0]}  (pins originally validated on 3.11; 3.12 wheels confirmed working)")

# ---------- package checks ----------
try:
    from packaging.specifiers import SpecifierSet
    from packaging.version import Version
    def meets(v, spec): return Version(v) in SpecifierSet(spec)
except ImportError:
    def meets(v, spec): return True

# (dist name, version spec, conda name if it differs)
PINS = [
    ("pyspark","==3.5.5",None), ("numpy",">=1.24,<2",None), ("pandas",">=2.0,<2.2",None),
    ("pyarrow",None,None), ("torch",">=2.4.0","pytorch"), ("transformers","==4.44.1",None),
    ("tokenizers","==0.19.0",None), ("datasets","==2.16.1",None), ("accelerate",">=0.31.0",None),
    ("peft","==0.10.0",None), ("optuna","==4.0.0",None), ("evaluate","==0.4.1",None),
    ("scikit-learn","==1.4.0",None), ("scipy","==1.12.0",None), ("xgboost","==2.0.3",None),
    ("dask","==2024.1.1",None), ("meds","==0.3.3",None), ("meds_reader","==0.1.9",None),
    ("femr",">=0.2.0,<=0.2.4",None), ("openai","==1.54.3",None), ("lightgbm",None,None),
    ("polars",None,None), ("wandb",">=0.17.8",None), ("packaging",">=23.2",None),
    ("pyarrow_hotfix",None,"pyarrow-hotfix"), ("foundry-transforms-lib-python",None,None),
    ("fast-ml","==3.68",None), ("Jinja2","==3.1.3","jinja2"), ("networkx",">=3.2.1",None),
    ("Pillow",">=10.3.0","pillow"), ("python-dateutil","==2.8.2",None), ("PyYAML","==6.0.1","pyyaml"),
    ("tqdm",">=4.66.1",None), ("Werkzeug","==3.0.1","werkzeug"), ("setuptools","<80.0.0",None),
    ("dill","<0.3.8",None), ("multiprocess","==0.70.15",None),
    ("fsspec","<=2023.10.0",None), ("huggingface-hub","<1.0","huggingface_hub"),
]
PIP_ONLY_LIKELY = {"meds","meds_reader","femr","fast-ml"}   # probably absent from conda channels

to_install = []   # (spec string for maestro, is_pip_only_likely)
print("=== packages ===")
for dist, spec, conda_name in PINS:
    cname = conda_name or dist
    try:
        v = md.version(dist)
        if v is None:                      # mangled dist-info (aborted install)
            raise md.PackageNotFoundError(dist)
        good = spec is None or meets(v, spec)
        check(dist, good, f"{v}" + (f"  -> {cname}{spec}" if not good else ""))
        if not good: to_install.append((cname + (spec or ""), dist in PIP_ONLY_LIKELY))
    except Exception:
        check(dist, False, f"missing/broken -> {cname}{spec or ''}")
        to_install.append((cname + (spec or ""), dist in PIP_ONLY_LIKELY))

TRANSITIVE = ["regex","joblib","threadpoolctl","filelock","fsspec","dill","multiprocess",
              "xxhash","huggingface_hub","safetensors","aiohttp","psutil","requests"]
miss = [t for t in TRANSITIVE if importlib.util.find_spec(t) is None]
check("transitive deps", not miss,
      f"missing: {', '.join(miss)} (should arrive as deps of the batch)" if miss else "all present")

# ---------- java / spark ----------
print(); print("=== java / spark ===")
def run_quiet(cmd):
    try: return subprocess.run(cmd, capture_output=True, text=True)
    except FileNotFoundError: return None
r = run_quiet(["java","-version"])
_out = ((r.stderr or "") + (r.stdout or "")) if r else ""
_lines = [l for l in _out.splitlines() if l.strip() and not l.startswith("Picked up")]
line = next((l for l in _lines if "version" in l.lower()), _lines[0] if _lines else "java binary not found")
good_java = bool(r) and '"17' in line
check("openjdk 17", good_java, line)
if not good_java: to_install.append(("openjdk==17.0.18", False))
r = run_quiet(["which","spark-submit"])
check("spark-submit on PATH", bool(r and r.stdout.strip()),
      (r.stdout.strip() if r and r.stdout.strip() else "missing (arrives with pyspark==3.5.5)"))
try:
    import pyspark
    cores = list((Path(pyspark.__file__).parent/"jars").glob("spark-core_*.jar"))
    check("pyspark jars coherent", len(cores)==1 and "3.5.5" in cores[0].name,
          cores[0].name if cores else "no spark-core jar")
except ModuleNotFoundError:
    check("pyspark jars coherent", False, "pyspark not installed yet")
except Exception as e:
    check("pyspark jars coherent", False, repr(e))

# ---------- local cehr packages (folders, never pip) ----------
print(); print("=== local cehr packages ===")
LOCAL = {"cehrbert_data": "/home/user/repo/cehrbert_data-main/src",
         "cehrgpt":       "/home/user/repo/cehrgpt-main/src",
         "cehrbert":      "/home/user/repo/cehrbert-main"}
for mod, root in LOCAL.items():
    spec = importlib.machinery.PathFinder().find_spec(mod, [root])
    check(f"local {mod}", bool(spec and spec.origin), (spec.origin if spec else f"missing under {root}"))
shadows = []
for d in ("cehrbert-data","cehrbert","cehrgpt"):
    try: md.distribution(d); shadows.append(d)
    except md.PackageNotFoundError: pass
check("no pip-installed cehr* shadows", not shadows,
      f"{shadows} installed -> REMOVE via GUI" if shadows else "")
rogue = Path("/home/user/repo/cehrbert_data")
check("no rogue /home/user/repo/cehrbert_data", not rogue.exists(),
      "exists - move it aside, it can shadow imports" if rogue.exists() else "")

# ---------- data dirs (idempotent) ----------
for d in ["omop_synthea","cehrgpt_output_full","model_run_full"]:
    os.makedirs(f"/home/user/repo/{d}", exist_ok=True)

# ---------- AUTO-INSTALL via Maestro's own CLI (same command the GUI runs) ----------
def maestro_cmd(args, label):
    cmd = ["maestro", "env"] + args
    print(f"\n>>> {label}:\n    {' '.join(cmd)}", flush=True)
    try:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             stdin=subprocess.DEVNULL, text=True, bufsize=1)
    except FileNotFoundError:
        print("    'maestro' CLI not found on PATH"); return False
    for ln in p.stdout: print(ln, end="")
    rc = p.wait(); print(f"    rc={rc}")
    return rc == 0

def maestro_install(specs, label, mode="conda"):
    if not specs: return True
    return maestro_cmd([mode, "install"] + specs, label)

# The 4 niche pkgs must NEVER live in the managed pip section: meds_reader declares
# pandas>=2.2 and femr declares open-ended transformers/datasets/sklearn/scipy, so every
# `maestro env install` pip phase force-reinstalls them WITH deps and clobbers our pins.
# The original working env installed them `pip --no-deps` - we reproduce exactly that:
# a no-deps layer applied AFTER each sync, never written into the spec.
NICHE_NO_DEPS = ["meds==0.3.3", "meds_reader==0.1.9", "femr==0.2.4", "fast-ml==3.68"]
REPAIR_EXACT  = {"pandas": "pandas==2.1.4", "transformers": "transformers==4.44.1",
                 "tokenizers": "tokenizers==0.19.0", "datasets": "datasets==2.16.1",
                 "scikit-learn": "scikit-learn==1.4.0", "scipy": "scipy==1.12.0",
                 "Jinja2": "jinja2==3.1.3", "python-dateutil": "python-dateutil==2.8.2",
                 "PyYAML": "pyyaml==6.0.1", "numpy": "numpy==1.26.4",
                 "dill": "dill==0.3.7", "multiprocess": "multiprocess==0.70.15",
                 "fsspec": "fsspec==2023.10.0", "huggingface-hub": "huggingface-hub==0.24.7"}

def pip_no_deps(specs, force=False):
    if not specs: return True
    cmd = ([sys.executable, "-m", "pip", "install", "--no-deps"]
           + (["--force-reinstall"] if force else []) + specs)
    print(f"\n>>> pip --no-deps{' --force-reinstall' if force else ''}: {' '.join(specs)}", flush=True)
    p = subprocess.run(cmd, capture_output=True, text=True)
    print((p.stdout + p.stderr).strip()[-1200:])
    print(f"    rc={p.returncode}")
    return p.returncode == 0

def broken_pins():
    importlib.invalidate_caches()
    out = []
    for dist, spec, conda_name in PINS:
        if dist in REPAIR_EXACT and spec:
            try:
                v = md.version(dist)
                if v is None or not meets(v, spec): out.append(REPAIR_EXACT[dist])
            except Exception:
                out.append(REPAIR_EXACT[dist])
    return out

print(); print("="*64)
if to_install and AUTO_INSTALL:
    # 0) pull the niche pkgs OUT of the managed pip section (harmless no-op if absent)
    maestro_cmd(["pip", "uninstall", "meds", "meds_reader", "femr", "fast-ml"],
                "removing niche pkgs from managed pip section (their deps clobber the pins)")
    # 1) conda batch for everything else
    main_specs = [s for s, piponly in to_install if not piponly]
    ok_main = maestro_install(main_specs, f"installing main batch ({len(main_specs)} specs, one solve)")
    # 1b) pre-sync heal: fix stale pip-layer pins (femr-era residue like dill==0.4.1 that
    #     contradicts datasets 2.16.1), then make a benign spec op so Hawk re-snapshots the
    #     lockfile from the now-healthy layer - otherwise the sync replay is unsolvable
    pre = broken_pins()
    if pre:
        print(f"\npre-sync repair of stale pip-layer pins: {pre}")
        pip_no_deps(pre, force=True)
        maestro_cmd(["pip", "install", "setuptools<80.0.0"], "re-snapshot pip lockfile (benign spec op)")
    # 2) full sync: Hawk materializes the env from the (now niche-free) spec
    ok_sync = maestro_cmd(["install"], "final environment sync")
    # 3) niche pkgs layered on top with --no-deps (mirrors the original working env)
    pip_no_deps(NICHE_NO_DEPS)
    # 4) repair pass: force back any pinned package the sync's pip phase clobbered
    broken = broken_pins()
    if broken:
        print(f"\nrepairing clobbered pins: {broken}")
        pip_no_deps(broken, force=True)
    print(); print("="*64)
    if not ok_sync:
        print("Final 'maestro env install' sync FAILED - read the output above before continuing.")
    if not ok_main:
        print("Main batch FAILED - read the solver output above.")
    print("\nNow RESTART THE KERNEL and re-run this cell until everything is OK.")
elif to_install:
    print(f"AUTO_INSTALL=False - {len(to_install)} item(s) to fix:")
    for s, _ in to_install: print(f"  - {s}")
elif BAD:
    print("No installs needed, but some checks failed - see FIX lines above.")
else:
    print("Environment matches target - proceed to the pipeline cells.")


In [ ]:
import subprocess
def maestro(*a):
    p=subprocess.Popen(["maestro","env",*a],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,stdin=subprocess.DEVNULL,text=True,bufsize=1)
    [print(l,end="") for l in p.stdout]; print("  rc=",p.wait())
maestro("conda","install","pytorch>=2.4")   # reset pytorch back to a plain spec

In [ ]:
import sys, subprocess
open("/tmp/pins.txt","w").write("""numpy==1.26.4
                                    jinja2==3.1.3
                                    fsspec==2023.10.0
                                    pandas==2.1.4
                                    transformers==4.44.1
                                    tokenizers==0.19.0
                                    datasets==2.16.1
                                    scikit-learn==1.4.0
                                    scipy==1.12.0
                                    huggingface-hub==0.24.7
                                    dill==0.3.7
                                    multiprocess==0.70.15
                                    """)

r = subprocess.run([sys.executable,"-m","pip","install","torch==2.12.0","-c","/tmp/pins.txt"], capture_output=True, text=True)
print(r.stdout[-2500:]); 
if r.returncode: print("ERR:\n", r.stderr[-2000:])

In [ ]:
import os, sys, subprocess

print("="*64, "\n1) nvidia-smi (driver + GPU present?)\n" + "="*64)
subprocess.run(["bash","-c","nvidia-smi || echo 'NO GPU / nvidia-smi missing'"])

print("\n" + "="*64, "\n2) torch in THIS kernel\n" + "="*64)
try:
    import torch
    print("torch", torch.__version__, "| built for CUDA", torch.version.cuda)
    ok = torch.cuda.is_available(); print("cuda available:", ok)
    if ok:
        print("count:", torch.cuda.device_count(), "| name:", torch.cuda.get_device_name(0))
        print("capability:", torch.cuda.get_device_capability(0), "| bf16:", torch.cuda.is_bf16_supported())
        free, tot = torch.cuda.mem_get_info(); print(f"GPU mem: {free/1e9:.1f} free / {tot/1e9:.1f} GB")
        import time; x = torch.randn(8192, 8192, device="cuda"); torch.cuda.synchronize()
        t=time.time()
        for _ in range(10): y = x @ x
        torch.cuda.synchronize(); print(f"matmul on GPU: OK ({time.time()-t:.3f}s for 10x 8192^3)")
    else:
        print(">>> kernel torch does NOT see the GPU")
except Exception as e:
    print("torch failed:", repr(e))

print("\n" + "="*64, "\n3) does the TRAINING subprocess env see the GPU?\n" + "="*64)
PKG = ["/home/user/repo/cehrbert-main","/home/user/repo/cehrgpt-main/src","/home/user/repo/cehrbert_data-main/src"]
probe = "import torch; a=torch.cuda.is_available(); print('cuda:', a, '| dev:', torch.cuda.get_device_name(0) if a else None)"
for label, ld in [("WITH conda-lib prepend (what training uses)", f"{sys.prefix}/lib:" + os.environ.get("LD_LIBRARY_PATH","")),
                  ("WITHOUT conda-lib prepend",                   os.environ.get("LD_LIBRARY_PATH",""))]:
    e = os.environ.copy(); e.pop("CUDA_VISIBLE_DEVICES", None)
    e["LD_LIBRARY_PATH"] = ld; e["PYTHONPATH"] = ":".join(PKG + [e.get("PYTHONPATH","")])
    r = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True, env=e)
    print(f"  {label}:\n    {r.stdout.strip() or r.stderr.strip()[-300:]}")

In [ ]:
import subprocess, sys, os
env = os.environ.copy()
env["LD_LIBRARY_PATH"] = f"{sys.prefix}/lib:" + env.get("LD_LIBRARY_PATH", "")
env["PYTHONPATH"] = ":".join([
    "/home/user/repo/cehrbert-main",
    "/home/user/repo/cehrgpt-main/src",
    "/home/user/repo/cehrbert_data-main/src",
    env.get("PYTHONPATH", ""),
])
r = subprocess.run([sys.executable, "-c", """
import cehrbert_data; print('cehrbert_data', cehrbert_data.__file__)
import cehrbert; print('cehrbert', cehrbert.__file__)
import cehrgpt; print('cehrgpt', cehrgpt.__file__)
from cehrbert.data_generators.hf_data_generator import meds_utils
import cehrgpt.runners.hf_cehrgpt_pretrain_runner
print('full import chain OK')
"""], capture_output=True, text=True, env=env)
print("STDOUT:")
print(r.stdout)
print("STDERR:")
print(r.stderr[-5000:])

## §1 Session patches (revert on every wipe re apply before running anything)

In [ ]:
# robustness patch: let the tokenizer survive malformed/negative time tokens like "i-D-6"
patch_path = "/home/user/repo/cehrgpt-main/src/cehrgpt/gpt_utils.py"
MARK = "robustness patch: tolerate malformed time tokens"
override = f'''

# {MARK}  (overrides extract_time_interval_in_days for tokens like "i-D-6")
import re as _re_patch
def extract_time_interval_in_days(token):  # noqa: F811
    try:
        return int(token.split("-")[1][1:])            # original fast path: "i-D6" -> 6
    except (IndexError, ValueError):
        m = _re_patch.search(r"-?\\d+$", str(token))     # trailing signed int: "i-D-6" -> -6
        return max(0, int(m.group())) if m else 0        # clamp negatives/unparseable to 0
'''

src = open(patch_path).read()
if MARK not in src:
    with open(patch_path, "a") as f: f.write(override)
    print("patched:", patch_path)
else:
    print("already patched")

In [ ]:
import datasets, os, sys, subprocess
tbl = os.path.join(os.path.dirname(datasets.__file__), "table.py")
MARK = "NaN-safe integer cast patch"
if MARK not in open(tbl).read():
    open(tbl, "a").write('''

# === NaN-safe integer cast patch ===
import pyarrow as _pa_safe, pyarrow.compute as _pc_safe
_orig_array_cast_safe = array_cast
def array_cast(array, pa_type, *args, **kwargs):  # noqa: F811
    try:
        if _pa_safe.types.is_integer(pa_type) and _pa_safe.types.is_floating(array.type):
            m = _pc_safe.or_(_pc_safe.is_null(array), _pc_safe.is_nan(array))
            array = _pc_safe.if_else(m, _pa_safe.scalar(0, type=array.type), array)
    except Exception:
        pass
    return _orig_array_cast_safe(array, pa_type, *args, **kwargs)
''')
    print("patched", tbl)
else:
    print("already patched")

# verify in a fresh subprocess (exactly what the workers load)
probe = """
from datasets.table import array_cast, cast_array_to_feature
from datasets.features import Sequence, Value
import pyarrow as pa
leaf = array_cast(pa.array([1.0, float('nan'), 3.0]), pa.int32()).to_pylist()
try: nested = cast_array_to_feature(pa.array([[1.0,float('nan')],[None,4.0]]), Sequence(Value('int32'))).to_pylist()
except Exception as e: nested = 'ERR:'+repr(e)[:40]
assert leaf == [1,0,3], leaf
print('CAST_SAFE_OK leaf=', leaf, 'nested=', nested)
"""
r = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
print(r.stdout.strip() or ("FAILED:\n" + r.stderr[-600:]))

In [ ]:
# cache get_vocab so it's built once per worker, not rebuilt per record
tok = "/home/user/repo/cehrgpt-main/src/cehrgpt/models/tokenization_hf_cehrgpt.py"
if "speed patch: memoize get_vocab" not in open(tok).read():
    open(tok, "a").write('''

# speed patch: memoize get_vocab (vocab is immutable during prep; avoids rebuilding it per record)
def _cehrgpt_get_vocab_cached(self):
    v = getattr(self, "_vocab_cache", None)
    if v is None:
        v = self._tokenizer.get_vocab(); self._vocab_cache = v
    return v
CehrGptTokenizer.get_vocab = _cehrgpt_get_vocab_cached
''')
    print("patched get_vocab (memoized)")
else:
    print("get_vocab already memoized")

In [ ]:
# generation patch A: tokenizer encode() must receive List[str]
src=open(TOKENIZER_PY).read()
old="encoded = self._tokenizer.encode(concept_ids, is_pretokenized=True)"
new="encoded = self._tokenizer.encode([str(c) for c in concept_ids], is_pretokenized=True)"
if new in src: print("encode(): already patched")
elif old in src: open(TOKENIZER_PY,"w").write(src.replace(old,new)); print("encode(): patched -> List[str]")
else: print("encode(): target line not found — check the file")

In [ ]:
# generation patch B: sanitize None in age/time wrappers
import re
src=open(GPT_UTILS_PY).read(); n_tot=0
for fn in ["_orig_construct_age_sequence","_orig_construct_time_sequence"]:
    pat=re.compile(rf"({re.escape(fn)}\()concept_ids([,)])")
    src,n=pat.subn(r"\1[('0' if c is None else str(c)) for c in concept_ids]\2", src); n_tot+=n
open(GPT_UTILS_PY,"w").write(src)
print(f"age/time sanitize: patched {n_tot} call(s)" + ("  <-- 0 means already patched or pattern changed" if n_tot==0 else ""))

## §2 Stage OMOP tables from Foundry

In [ ]:
from foundry.transforms import Dataset
from pathlib import Path
import shutil, time, traceback

# Foundry dataset -> (OMOP canonical name, expected parquet count from the full set)
DATASETS = {
    "copd_concept_parquet":              ("concept",              4),
    "copd_ancestor_concept_copd":        ("concept_ancestor",     5),
    "copd_relationship_concept_parquet": ("concept_relationship", 3),
    "copd_condition_occurrence_parquet": ("condition_occurrence", 200),
    "copd_drug_exposure_parquet":        ("drug_exposure",        526),
    "copd_measurement_parquet":          ("measurement",          2674),
    "copd_observation_parquet":          ("observation",          236),
    "copd_procedure_occurrence_parquet": ("procedure_occurrence", 71),
    "copd_visit_occurrence_parquet":     ("visit_occurrence",     67),
    "person":                            ("person",               4),
    "copd_death_parquet":                ("death",                1),
}

base_dir = Path("/home/user/repo/omop_synthea")
def log(m): print(f"[{time.strftime('%H:%M:%S')}] {m}", flush=True)

for fd_name, (omop_name, expected) in DATASETS.items():
    out_dir = base_dir / omop_name
    have = len(list(out_dir.glob("*.parquet"))) if out_dir.is_dir() else 0
    if have >= expected:
        log(f"SKIP  {omop_name:24s} ({have} parquet files present)"); continue
    log(f"STAGE {omop_name:24s} have {have}, need {expected} -- downloading {fd_name}")
    if out_dir.is_file(): out_dir.unlink()
    elif out_dir.exists(): shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True)
    try:
        downloaded = Dataset.get(fd_name).files().download()
        n = size = 0
        for src_path in downloaded.values():
            src = Path(src_path)
            if src.suffix == ".parquet":
                shutil.copy(src, out_dir / src.name); n += 1; size += src.stat().st_size
        (out_dir / "_SUCCESS").touch()
        log(f"      done: {n} parquet files, {size/1e9:.2f} GB")
    except Exception as e:
        log(f"      FAILED: {type(e).__name__}: {e}"); traceback.print_exc()


import os
BASE = "/home/user/repo/omop_synthea"
EXPECT = {"concept":4,"concept_ancestor":5,"concept_relationship":3,"condition_occurrence":200,
          "drug_exposure":526,"measurement":2674,"observation":236,"procedure_occurrence":71,
          "visit_occurrence":67,"person":4, **({"death":1} if INCLUDE_DEATH else {})}
ok = True
for t, want in EXPECT.items():
    d = os.path.join(BASE, t)
    n = sum(f.endswith(".parquet") for _,_,fs in os.walk(d) for f in fs) if os.path.isdir(d) else 0
    print(f"  {'OK  ' if n >= want else 'FAIL'}  {t:24s} {n:5d} / {want}")
    ok = ok and n >= want
print("\nREADY" if ok else "\nNOT READY -- re-run the download cell")



## §3 Measurement prep

1. disk headroom → 2. build real visit links → 3. **swap that output into `measurement/`** → 4. back-fill only genuinely absent columns → 5. clear stale caches.

In [ ]:
import shutil, glob, os
OMOP="/home/user/repo/omop_synthea"
free=shutil.disk_usage(OMOP)[2]/1e9
msize=sum(os.path.getsize(p) for p in glob.glob(f"{OMOP}/measurement/*.parquet"))/1e9
print(f"free disk: {free:.0f} GB | measurement size: {msize:.0f} GB | need ~{msize+10:.0f} GB free to rewrite safely")
print("OK to proceed" if free > msize+10 else "TIGHT — tell me and we'll do a chunked in-place rewrite instead")

In [ ]:
import subprocess, os, sys, textwrap, time, re
from pathlib import Path
OMOP = os.environ["OMOP_DIR"]
SCRIPT = "/home/user/repo/_fix_meas_visits.py"
Path(SCRIPT).write_text(textwrap.dedent(f'''
    from pyspark.sql import SparkSession, functions as F
    spark = SparkSession.builder.appName("fix_meas_visits").config("spark.sql.session.timeZone","UTC").getOrCreate()
    OMOP = "{OMOP}"
    meas = spark.read.parquet(OMOP+"/measurement").drop("visit_occurrence_id")
    meas = meas.withColumn("_mid", F.monotonically_increasing_id())
    vis = spark.read.parquet(OMOP+"/visit_occurrence").select(
        "person_id","visit_occurrence_id",
        F.col("visit_start_date").cast("date").alias("vs"),
        F.coalesce(F.col("visit_end_date"), F.col("visit_start_date")).cast("date").alias("ve"))
    key = meas.select("_mid","person_id", F.col("measurement_date").cast("date").alias("d"))
    matched = (key.join(vis, (key.person_id==vis.person_id) & (key.d>=vis.vs) & (key.d<=vis.ve))
                  .groupBy("_mid").agg(F.min("visit_occurrence_id").alias("visit_occurrence_id")))
    out = meas.join(matched, "_mid", "left").drop("_mid")
    n = meas.count(); m = out.where(F.col("visit_occurrence_id").isNotNull()).count()
    print(f"MEAS_MATCH {{m}} of {{n}} ({{100.0*m/n:.1f}}%) matched to a visit", flush=True)
    out.write.mode("overwrite").parquet(OMOP+"/measurement_withvisit")
    print("WROTE measurement_withvisit", flush=True)
'''))
JVM=("--add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED "
     "--add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED")
env=os.environ.copy(); env["JAVA_TOOL_OPTIONS"]=env["_JAVA_OPTIONS"]=JVM; env["JAVA_HOME"]=sys.prefix
env["PATH"]=f"{sys.prefix}/bin:"+env.get("PATH",""); env["LD_LIBRARY_PATH"]=f"{sys.prefix}/lib:"+env.get("LD_LIBRARY_PATH","")
os.makedirs("/home/user/repo/spark_tmp", exist_ok=True)
cmd=["spark-submit","--master","local[16]","--driver-memory","48g",
     "--conf","spark.sql.shuffle.partitions=600","--conf","spark.sql.adaptive.enabled=true",
     "--conf","spark.local.dir=/home/user/repo/spark_tmp",
     "--conf","spark.sql.autoBroadcastJoinThreshold=-1",
     "--driver-java-options",JVM, SCRIPT]
log=Path("/home/user/repo/logs")/f"meas_visits-{time.strftime('%H%M%S')}.log"
print("log:",log); t0=time.time(); n=0
with open(log,"w") as lf:
    p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
    for line in p.stdout:
        lf.write(line); n+=1
        if re.search(r'MEAS_MATCH|WROTE|ERROR|Exception|Traceback',line): print(line,end="")
        elif n%500==0: print(f"  ...{n} lines, {(time.time()-t0)/60:.1f} min",flush=True)
rc=p.wait(); print(f"\nrc={rc} ({(time.time()-t0)/60:.1f} min)")
if rc!=0: print("".join(open(log).readlines()[-20:]))

In [ ]:
# SWAP the visit-linked measurement into place (previously orphaned!)
import glob, os, shutil, subprocess
import pyarrow.parquet as pq
SRC=f"{OMOP_DIR}/measurement_withvisit"; DST=f"{OMOP_DIR}/measurement"
assert glob.glob(f"{SRC}/*.parquet"), f"{SRC} missing — run the visit-link cell above first"
BAK=f"{OMOP_DIR}/measurement_orig"
if not os.path.exists(BAK):
    shutil.move(DST,BAK); print("original measurement -> measurement_orig")
else:
    shutil.rmtree(DST, ignore_errors=True)
os.makedirs(DST, exist_ok=True)
subprocess.run(["bash","-lc", f"cp {SRC}/*.parquet {DST}/"], check=True)
f0=sorted(glob.glob(f"{DST}/*.parquet"))[0]
t=pq.read_table(f0, columns=["visit_occurrence_id"])["visit_occurrence_id"]
print(f"swapped in {len(glob.glob(f'{DST}/*.parquet'))} files | "
      f"visit_occurrence_id non-null in sample: {len(t)-t.null_count:,}/{len(t):,}")

In [ ]:
import glob, pyarrow as pa, pyarrow.parquet as pq
mfiles = sorted(glob.glob("/home/user/repo/omop_synthea/measurement/*.parquet"))
ADD = {"measurement_datetime": pa.timestamp("us"),
       "visit_occurrence_id":  pa.int64(),
       "unit_source_value":    pa.string()}
done = 0
for f in mfiles:
    t = pq.read_table(f)
    if set(ADD) <= set(t.column_names): done += 1; continue   # already patched
    cols = {n: t[n] for n in t.column_names}
    for name, typ in ADD.items():
        if name not in cols: cols[name] = pa.nulls(t.num_rows, typ)
    pq.write_table(pa.table(cols), f)
    done += 1
    if done % 300 == 0: print(f"  ...{done}/{len(mfiles)}")
print(f"patched {done} measurement files with {list(ADD)}")
print("new columns:", pq.read_schema(mfiles[0]).names)

In [ ]:
import shutil, glob, os
OMOP="/home/user/repo/omop_synthea"
for d in glob.glob(f"{OMOP}/processed_*"):
    n=len(glob.glob(f"{d}/*.parquet"))
    if n==0:
        shutil.rmtree(d); print("removed empty cache:", os.path.basename(d))
    else:
        print("keeping", os.path.basename(d), f"({n} files)")
print("done — now re-run the qualified-concept-list cell")

## §4 Qualified concept list
Hard fails if the value columns are missing a retrain is pointless without them.

In [ ]:
import subprocess, os, sys, textwrap, shutil, time, re, glob
from pathlib import Path
import pyarrow.parquet as pq

OMOP = os.environ["OMOP_DIR"]
assert glob.glob(f"{OMOP}/concept/*.parquet") and glob.glob(f"{OMOP}/condition_occurrence/*.parquet")

# --- preflight: measurement must be staged AND carry the columns the handler needs ---
mfiles = glob.glob(f"{OMOP}/measurement/*.parquet")
assert mfiles, "measurement not staged — run the Step 0 staging cell first"
mcols = set(pq.read_schema(mfiles[0]).names)
date_ok = bool({"measurement_date", "measurement_datetime"} & mcols)
val_ok  = "value_as_number" in mcols
cat_ok  = "value_as_concept_id" in mcols
print("measurement columns:", sorted(mcols))
print(f"  measurement_concept_id: {'OK' if 'measurement_concept_id' in mcols else 'MISSING'}"
      f" | date: {'OK' if date_ok else 'MISSING'} | value_as_number: {'OK' if val_ok else 'MISSING'}")
if not ("measurement_concept_id" in mcols and date_ok):
    raise SystemExit("measurement export lacks concept_id/date — fix the export before adding it (this is why it was excluded).")
print(f"  value_as_concept_id: {'OK' if cat_ok else 'MISSING'}")
if INCLUDE_VALUES and not val_ok:
    raise SystemExit("STOP: value_as_number missing — a values retrain cannot work. Fix the export first.")
if INCLUDE_VALUES and not cat_ok:
    print("  ⚠ no value_as_concept_id — categorical results (Positive/Negative) will stay empty")

shutil.rmtree(f"{OMOP}/qualified_concept_list", ignore_errors=True)

LAUNCHER = "/home/user/repo/_cehr_launcher_cl.py"
Path(LAUNCHER).write_text(textwrap.dedent(r'''
    import sys, importlib, runpy
    REAL = ["/home/user/repo/cehrbert_data-main/src","/home/user/repo/cehrbert-main","/home/user/repo/cehrgpt-main/src"]
    for p in reversed(REAL):
        while p in sys.path: sys.path.remove(p)
        sys.path.insert(0, p)
    for m in [m for m in list(sys.modules) if m=="cehrbert_data" or m.startswith("cehrbert_data.")]:
        del sys.modules[m]
    importlib.invalidate_caches()
    APP = "/home/user/repo/cehrbert_data-main/src/cehrbert_data/apps/generate_included_concept_list.py"
    sys.argv = [APP] + sys.argv[1:]
    runpy.run_path(APP, run_name="__main__")
'''))

JVM_OPTS = ("--add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED "
            "--add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED "
            "--add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED")
LOG4J2_OPT = "-Dlog4j2.configurationFile=file:/home/user/repo/log4j2.properties"
PKG = "/home/user/repo/cehrbert_data-main/src:/home/user/repo/cehrbert-main:/home/user/repo/cehrgpt-main/src"
env = os.environ.copy()
env["JAVA_TOOL_OPTIONS"] = env["_JAVA_OPTIONS"] = JVM_OPTS
env["JAVA_HOME"] = sys.prefix
env["PATH"] = f"{sys.prefix}/bin:" + env.get("PATH", "")
env["LD_LIBRARY_PATH"] = f"{sys.prefix}/lib:" + env.get("LD_LIBRARY_PATH", "")
env["PYTHONPATH"] = PKG + ":" + env.get("PYTHONPATH", "")

os.makedirs("/home/user/repo/spark_tmp", exist_ok=True)
cmd = ["spark-submit", "--master", "local[16]", "--driver-memory", "48g",
       "--conf", "spark.sql.shuffle.partitions=200", "--conf", "spark.sql.adaptive.enabled=true",
       "--conf", "spark.local.dir=/home/user/repo/spark_tmp",
       "--conf", "spark.serializer=org.apache.spark.serializer.KryoSerializer",
       "--driver-java-options", f"{JVM_OPTS} {LOG4J2_OPT}",
       LAUNCHER, "-i", OMOP, "-o", OMOP,
       "--min_num_of_patients", "100",
       "--ehr_table_list", "condition_occurrence", "drug_exposure", "procedure_occurrence", "measurement"]

LOGDIR = Path("/home/user/repo/logs"); LOGDIR.mkdir(exist_ok=True)
log_path = LOGDIR / f"qcl_full-{time.strftime('%Y%m%d-%H%M%S')}.log"
KEEP = re.compile(r'ERROR|Exception|Traceback|FAILED|py4j\.protocol|File "', re.I)
print(f"logfile: {log_path}\n")
n = 0; t0 = time.time()
with open(log_path, "w") as lf:
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout:
        lf.write(line); n += 1
        if KEEP.search(line): print(line, end="")
        elif n % 500 == 0: print(f"  ... {n} lines, {(time.time()-t0)/60:.1f} min", flush=True)
rc = p.wait()
print(f"\nrc={rc}  ({n} lines, {(time.time()-t0)/60:.1f} min)")
if rc != 0:
    print("".join(open(log_path).readlines()[-30:]))
else:
    fs = glob.glob(f"{OMOP}/qualified_concept_list/*.parquet")
    print(f"qualified_concept_list -> {len(fs)} files | rows: {sum(pq.ParquetFile(f).metadata.num_rows for f in fs):,}" if fs else "EMPTY")

## §5 Build `patient_sequence`

Run the **flag check** first confirm the arg names match your `cehrbert_data` version before launching a multi hour Spark job.

In [ ]:
# verify the app's flags BEFORE the long run
import subprocess, sys, os
env=os.environ.copy()
env["PYTHONPATH"]=":".join([PKG_CEHRBERT_DATA,PKG_CEHRBERT,PKG_CEHRGPT])
h=subprocess.run([sys.executable,APP_SEQ,"--help"],capture_output=True,text=True,env=env)
out=(h.stdout+h.stderr)
print(out[:4000])
for f in ["--domain_table_list","--include_concept_list","--gpt_patient_sequence",
          "--is_new_patient_representation","--include_inpatient_hour_token","--aggregate_by_hour",
          "--should_construct_artificial_visits","--disconnect_problem_list_records"]:
    print(f"  {'OK ' if f in out else 'ABSENT'}  {f}")

In [ ]:
# ---- BUILD patient_sequence (Spark; hours) ----
import subprocess, os, sys, textwrap, shutil, time, re, glob
from pathlib import Path
import pyarrow.parquet as pq

assert glob.glob(f"{OMOP_DIR}/qualified_concept_list/*.parquet"), "run §4 first"
shutil.rmtree(f"{OMOP_DIR}/processed_measurement", ignore_errors=True)   # stale cache breaks aggregate_by_hour
shutil.rmtree(PATIENT_SEQ, ignore_errors=True)

LAUNCH=f"{REPO}/_cehr_launcher_seq.py"
Path(LAUNCH).write_text(textwrap.dedent(f'''
    import sys, importlib, runpy
    REAL = ["{PKG_CEHRBERT_DATA}","{PKG_CEHRBERT}","{PKG_CEHRGPT}"]
    for p in reversed(REAL):
        while p in sys.path: sys.path.remove(p)
        sys.path.insert(0, p)
    for m in [m for m in list(sys.modules) if m=="cehrbert_data" or m.startswith("cehrbert_data.")]:
        del sys.modules[m]
    importlib.invalidate_caches()
    APP = "{APP_SEQ}"
    sys.argv = [APP] + sys.argv[1:]
    runpy.run_path(APP, run_name="__main__")
'''))

JVM=("--add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED "
     "--add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED "
     "--add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED")
env=os.environ.copy()
env["JAVA_TOOL_OPTIONS"]=env["_JAVA_OPTIONS"]=JVM; env["JAVA_HOME"]=sys.prefix
env["PATH"]=f"{sys.prefix}/bin:"+env.get("PATH","")
env["LD_LIBRARY_PATH"]=f"{sys.prefix}/lib:"+env.get("LD_LIBRARY_PATH","")
env["PYTHONPATH"]=":".join([PKG_CEHRBERT_DATA,PKG_CEHRBERT,PKG_CEHRGPT,env.get("PYTHONPATH","")])

# Spark tuning learned from the WindowExec OOM: fewer cores, more shuffle, skew join on
cmd=["spark-submit","--master","local[8]","--driver-memory","44g",
     "--conf","spark.memory.fraction=0.75",
     "--conf","spark.sql.shuffle.partitions=2000",
     "--conf","spark.sql.adaptive.enabled=true",
     "--conf","spark.sql.adaptive.skewJoin.enabled=true",
     "--conf","spark.sql.files.maxPartitionBytes=64m",
     "--conf",f"spark.local.dir={SPARK_TMP}",
     "--conf","spark.serializer=org.apache.spark.serializer.KryoSerializer",
     "--driver-java-options",f"{JVM} -Dlog4j2.configurationFile=file:{LOG4J}",
     LAUNCH,
     "-i",OMOP_DIR,"-o",DATA_DIR,
     "--domain_table_list","condition_occurrence","drug_exposure","procedure_occurrence","measurement",
     "--include_concept_list","--gpt_patient_sequence","--is_new_patient_representation",
     "--include_inpatient_hour_token","--aggregate_by_hour",
     "--should_construct_artificial_visits","--disconnect_problem_list_records"]
print("death table staged:", bool(glob.glob(f"{OMOP_DIR}/death/*.parquet")),
      "(DeathEventDecorator fires automatically when present)")

log=Path(LOG_DIR)/f"seqbuild-{time.strftime('%Y%m%d-%H%M%S')}.log"
KEEP=re.compile(r'ERROR|Exception|Traceback|FAILED|OutOfMemory|py4j\.protocol', re.I)
print("log:",log,"\n"); n=0; t0=time.time()
with open(log,"w") as lf:
    p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
    for line in p.stdout:
        lf.write(line); n+=1
        if KEEP.search(line): print(line,end="")
        elif n%500==0: print(f"  ... {n} lines, {(time.time()-t0)/60:.1f} min",flush=True)
rc=p.wait(); print(f"\nrc={rc}  ({(time.time()-t0)/60:.1f} min)")
if rc==0:
    fs=glob.glob(f"{PATIENT_SEQ}/*.parquet")
    print(f"patient_sequence: {len(fs)} files, {sum(pq.read_metadata(f).num_rows for f in fs):,} sequences")
else:
    print("".join(open(log).readlines()[-40:]))

In [ ]:
# VERIFY values + death actually landed in the new sequences
import glob, collections, pyarrow.parquet as pq
files=sorted(glob.glob(f"{PATIENT_SEQ}/*.parquet"))[:5]
assert files, "no patient_sequence — build failed"
tot=withnum=withcat=0; nnum=0; DEATH={"4306655","4304866","4267568","443768"}; dhit=0
for f in files:
    t=pq.read_table(f)
    cols=t.schema.names
    ci=t["concept_ids"].to_pylist()
    m=t["concept_value_masks"].to_pylist() if "concept_value_masks" in cols else [None]*t.num_rows
    nv=t["number_as_values"].to_pylist() if "number_as_values" in cols else [None]*t.num_rows
    cv=t["concept_as_values"].to_pylist() if "concept_as_values" in cols else [None]*t.num_rows
    for a,mm,vv,cc in zip(ci,m,nv,cv):
        tot+=1
        if mm and sum(int(x or 0) for x in mm)>0: withnum+=1
        if vv: nnum+=sum(1 for x in vv if x not in (None,0,0.0))
        if cc and any(x not in (None,"") for x in cc): withcat+=1
        if a and any(str(x) in DEATH for x in a): dhit+=1
print(f"{tot:,} sequences sampled")
print(f"  with numeric lab values : {withnum:,} ({100*withnum/tot:.0f}%)   total values {nnum:,}")
print(f"  with categorical values : {withcat:,} ({100*withcat/tot:.0f}%)")
print(f"  with a death concept    : {dhit:,}")
print("\n--> numeric % near 99 means values are IN. 0% means the build dropped them: STOP, do not train.")

## §6 Clean, filter, bank
fill nulls → filter to ≤`MAX_SEQ_LEN` tokens (values make sequences heavier; this is what stopped the prep OOM) → bank to Foundry.

In [ ]:
import glob, pyarrow as pa, pyarrow.parquet as pq, pyarrow.compute as pc

seq = "/home/user/repo/cehrgpt_output_full/patient_sequence"
files = sorted(glob.glob(seq + "/*.parquet"))
FILL = {"ages": 0.0, "dates": 0, "epoch_times": 0}   # numeric cols cehrbert casts to int; null -> 0

def fill_nulls(col, val):
    arr = col.combine_chunks()
    if pa.types.is_list(arr.type):
        return pa.ListArray.from_arrays(arr.offsets, pc.fill_null(arr.values, val)).cast(arr.type)
    if pa.types.is_large_list(arr.type):
        return pa.LargeListArray.from_arrays(arr.offsets, pc.fill_null(arr.values, val)).cast(arr.type)
    return pc.fill_null(arr, val)

for fi, f in enumerate(files):
    t = pq.read_table(f)
    new = {n: (fill_nulls(t[n], FILL[n]) if n in FILL else t[n]) for n in t.column_names}
    pq.write_table(pa.table(new, schema=t.schema), f)
    if fi % 100 == 0: print(f"  ...{fi}/{len(files)}")
        
print("filled ages/dates/epoch_times nulls in all files")

In [ ]:
# length filter before training (and re-run after any restore)
import glob, pyarrow.parquet as pq, pyarrow.compute as pc
kept=tot=0
for f in sorted(glob.glob(f"{PATIENT_SEQ}/*.parquet")):
    t=pq.read_table(f); tot+=t.num_rows
    sub=t.filter(pc.less_equal(t["num_of_concepts"], MAX_SEQ_LEN))
    pq.write_table(sub,f); kept+=sub.num_rows
print(f"filtered <= {MAX_SEQ_LEN} tokens: kept {kept:,} of {tot:,} ({100*kept/max(tot,1):.0f}%)")

In [ ]:
# bank the v3 sequences (create ALIAS_SEQ_V3 in the UI + restart kernel if it 404s)
import glob, os, subprocess
import pyarrow.parquet as pq
from foundry.transforms import Dataset
files=sorted(glob.glob(f"{PATIENT_SEQ}/*.parquet")); assert files
rows=sum(pq.read_metadata(f).num_rows for f in files)
try: Dataset.get(ALIAS_SEQ_V3).files()
except Exception as e: raise SystemExit(f"create '{ALIAS_SEQ_V3}' in the UI + restart kernel ({repr(e)[:60]})")
TAR=f"{REPO}/patient_sequence_v3.tar"
subprocess.run(["bash","-lc",f"cd {PATIENT_SEQ} && tar cf {TAR} . ; [ $? -le 1 ] && ls -lh {TAR}"],check=True)
Dataset.get(ALIAS_SEQ_V3).upload_file(TAR,"patient_sequence_v3.tar")
print(f"banked {rows:,} sequences -> {ALIAS_SEQ_V3}")

In [ ]:
# RESTORE v3 sequences after a wipe (skip if you just built them)
import glob, os, subprocess
import pyarrow.parquet as pq, pyarrow.compute as pc
from foundry.transforms import Dataset
subprocess.run(["bash","-lc",f"rm -rf {PATIENT_SEQ} && mkdir -p {PATIENT_SEQ}"],check=True)
tar=max([p for p in Dataset.get(ALIAS_SEQ_V3).files().download().values() if str(p).endswith(".tar")], key=os.path.getsize)
subprocess.run(["bash","-lc",f"tar xf {tar} -C {PATIENT_SEQ}"],check=True)
kept=tot=0
for f in sorted(glob.glob(f"{PATIENT_SEQ}/*.parquet")):     # re-apply the filter, a restore reverts it
    t=pq.read_table(f); tot+=t.num_rows
    sub=t.filter(pc.less_equal(t["num_of_concepts"], MAX_SEQ_LEN)); pq.write_table(sub,f); kept+=sub.num_rows
print(f"restored + re-filtered: {kept:,} of {tot:,} sequences")

## §7 Train (v3, **values on**)

GPU cleanup → checkpoint watcher → pre-flight → train.

In [ ]:
import os, gc, sys, subprocess, signal, time

def gpu_status(label):
    print(f"\n=== GPU {label} ===")
    subprocess.run(["nvidia-smi","--query-gpu=memory.used,memory.free,memory.total","--format=csv"])
    subprocess.run(["nvidia-smi","--query-compute-apps=pid,used_memory,process_name","--format=csv"])

gpu_status("before")

# 1) kill orphaned CEHR-GPT training subprocesses still holding VRAM (never this kernel)
my_pid = os.getpid()
pids = subprocess.run(["nvidia-smi","--query-compute-apps=pid","--format=csv,noheader"],
                      capture_output=True, text=True).stdout.split()
killed = []
for tok in pids:
    if not tok.strip().isdigit(): continue
    pid = int(tok)
    if pid == my_pid: continue
    try: name = open(f"/proc/{pid}/cmdline").read().replace("\x00"," ")
    except Exception: name = ""
    if "cehrgpt" in name.lower():
        try: os.kill(pid, signal.SIGKILL); killed.append(pid)
        except Exception as e: print("  couldn't kill", pid, e)
subprocess.run(["bash","-c","pkill -9 -f hf_cehrgpt_pretrain_runner 2>/dev/null; true"])
print("killed stray training procs:", killed or "none")

# 2) clear torch cache ONLY if this kernel already holds a CUDA context (don't create one on a fresh kernel)
if "torch" in sys.modules:
    for v in ["x","y","model","trainer","batch","out","logits"]:
        globals().pop(v, None)
    gc.collect()
    t = sys.modules["torch"]
    if t.cuda.is_available():
        t.cuda.empty_cache(); t.cuda.ipc_collect(); t.cuda.reset_peak_memory_stats()
        print(f"cleared kernel torch cache (reserved now {t.cuda.memory_reserved()/1e9:.2f} GB)")
else:
    print("torch not imported here — no kernel CUDA context to clear (ideal)")

time.sleep(2)
gpu_status("after")

In [ ]:
import subprocess, sys, os, time, textwrap
from pathlib import Path
WATCHER = "/home/user/repo/resume_watcher.py"
Path("/home/user/repo/logs").mkdir(parents=True, exist_ok=True)
Path(WATCHER).write_text(textwrap.dedent('''
    import time, glob, os, subprocess
    from foundry.transforms import Dataset
    OUTPUT_DIR = "/home/user/repo/model_run_full_real"
    TOKENIZER  = "/home/user/repo/model_run_full"
    PREP       = "/home/user/repo/cehrgpt_output_full/dataset_prepared_real"
    CKPT_DS, PREP_DS = "copd_cehrgpt_v3_resume", "copd_dataset_prepared_v3"
    INTERVAL = 120
    def latest():
        cks=[c for c in glob.glob(OUTPUT_DIR+"/checkpoint-*") if os.path.exists(os.path.join(c,"trainer_state.json"))]
        return max(cks, key=lambda c:int(c.split("-")[-1])) if cks else None
    last=None; prep_banked=False
    print(time.strftime("%H:%M:%S"),"watcher up",flush=True)
    while True:
        try:
            if not prep_banked and glob.glob(PREP+"/**/dataset_info.json", recursive=True):
                pt="/home/user/repo/dataset_prepared_full.tar"
                subprocess.run(["bash","-lc",f"cd {os.path.dirname(PREP)} && tar cf {pt} dataset_prepared_real"],check=True)
                Dataset.get(PREP_DS).upload_file(pt,"dataset_prepared_full.tar")
                prep_banked=True; print(time.strftime("%H:%M:%S"),"BANKED dataset_prepared",flush=True)
            ck=latest()
            if ck and ck!=last:
                t="/home/user/repo/resume.tar"; n=os.path.basename(ck)
                subprocess.run(["bash","-lc",f"tar cf {t} -C {os.path.dirname(TOKENIZER)} {os.path.basename(TOKENIZER)} -C {OUTPUT_DIR} {n}"],check=True)
                Dataset.get(CKPT_DS).upload_file(t,"resume.tar")
                last=ck; print(time.strftime("%H:%M:%S"),"BANKED",n,f"({os.path.getsize(t)/1e9:.2f} GB)",flush=True)
        except Exception as e:
            print(time.strftime("%H:%M:%S"),"WATCHER ERROR:",repr(e)[:200],flush=True)
        time.sleep(INTERVAL)
    '''))
subprocess.run(["bash","-lc","pkill -f resume_watcher.py 2>/dev/null; true"]); time.sleep(1)
logf=open("/home/user/repo/logs/watcher.log","a")
proc=subprocess.Popen([sys.executable,"-u",WATCHER],stdout=logf,stderr=subprocess.STDOUT,start_new_session=True,env=os.environ.copy())
print("launched PID",proc.pid); time.sleep(6)
print(subprocess.run(["bash","-lc","tail -8 /home/user/repo/logs/watcher.log"],capture_output=True,text=True).stdout)

In [ ]:
# ===================== PRE-FLIGHT run RIGHT before training =====================
import os, sys, glob, shutil, subprocess
from pathlib import Path

REPO = "/home/user/repo"
DATA, MODEL = f"{REPO}/cehrgpt_output_full", f"{REPO}/model_run_full"
PKG = [f"{REPO}/cehrbert-main", f"{REPO}/cehrgpt-main/src", f"{REPO}/cehrbert_data-main/src"]
checks = []
def chk(name, ok, detail=""):
    checks.append(ok); print(f"  {'✅' if ok else '❌'} {name:40s} {detail}")

print("=== 1) GPU ===")
try:
    import torch; g = torch.cuda.is_available()
    chk("CUDA GPU available", g, torch.cuda.get_device_name(0) if g else "FALSE → CPU = months, STOP")
except Exception as e: chk("torch import", False, repr(e)[:60])

print("=== 2) both gpt_utils patches on disk ===")
gu = open(f"{REPO}/cehrgpt-main/src/cehrgpt/gpt_utils.py").read()
chk("patch v1 (time tokens)", "robustness patch: tolerate malformed time tokens" in gu)
chk("patch v2 (age/time sanitize)", "robustness patch v2: sanitize age/time sequences" in gu)

print("=== 3) patch is LIVE in a training-style subprocess ===")
probe = f"""
import sys; sys.path[:0] = {PKG!r}
from cehrgpt.gpt_utils import construct_age_sequence, construct_time_sequence
import pyarrow as pa
a = construct_age_sequence(['Year:2000','Age:40','x'], [None, float('nan'), 41.0])
t = construct_time_sequence(['Year:2000','Age:40','x'], [None, float('nan'), 1.0e9])
bad = [x for x in a + t if x is None or x != x]
pa.array(a, type=pa.int32())                      # the exact op that crashed
print('SUBPROC_OK', a, 'no-nan' if not bad else 'BAD')
"""
r = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True,
                   env={**os.environ, "PYTHONPATH": ":".join(PKG)})
ok3 = "SUBPROC_OK" in r.stdout and "no-nan" in r.stdout
chk("subprocess runs PATCHED funcs, no NaN", ok3,
    (r.stdout.strip().splitlines()[-1] if r.stdout else r.stderr.strip()[-80:]))

print("=== 4) data + dirs ===")
ps = glob.glob(f"{DATA}/patient_sequence/*.parquet")
chk("patient_sequence files present", len(ps) >= 500, f"{len(ps)} files")
chk("model_run_full exists", Path(MODEL).exists(), MODEL)
prep = Path(f"{DATA}/dataset_prepared_real")
stale = prep.exists() and any(prep.rglob("*"))
chk("no stale dataset_prepared_real", not stale, "exists → set REBUILD_PREP=True" if stale else "clean")

print("=== 5) import chain (subprocess) ===")
r2 = subprocess.run([sys.executable, "-c",
   "import cehrbert_data, cehrbert, cehrgpt, cehrgpt.runners.hf_cehrgpt_pretrain_runner; print('IMPORTS_OK')"],
   capture_output=True, text=True, env={**os.environ, "PYTHONPATH": ":".join(PKG)})
chk("cehrgpt runner imports", "IMPORTS_OK" in r2.stdout, "" if "IMPORTS_OK" in r2.stdout else r2.stderr.strip()[-80:])

print("=== 6) checkpoint safety net ===")
w = subprocess.run(["bash","-lc","pgrep -af resume_watcher.py"], capture_output=True, text=True).stdout.strip()
chk("watcher process alive", bool(w), w.split(chr(10))[0] if w else "NOT running → a crash loses progress")
try:
    from foundry.transforms import Dataset
    Dataset.get("copd_cehrgpt_full_resume").files().download()
    chk("resume alias reachable", True, "copd_cehrgpt_full_resume")
except Exception as e:
    chk("resume alias reachable", "AliasNotDefined" not in str(e), repr(e)[:50])

print("=== 7) disk headroom ===")
free = shutil.disk_usage(REPO)[2] / 1e9
chk("free disk > 30 GB", free > 30, f"{free:.0f} GB free")

print("\n================ GO / NO-GO ================")
print("✅ ALL CLEAR — launch the training cell now." if all(checks)
      else "❌ STOP — resolve the ❌ line(s) above before launching.")

In [ ]:
import os, sys, shutil, re, time, subprocess, torch
from pathlib import Path

NUM_EPOCHS, MODEL_CTX_LEN, GPU_MAX_TOKENS, SAVE_STEPS = 3, 1024, 4096, 2000
RESUME, REBUILD_PREP, FRESH_OUTPUT = False, True, True
if RESUME: FRESH_OUTPUT = False

OMOP_DIR="/home/user/repo/omop_synthea"; CEHR_GPT_DATA_DIR="/home/user/repo/cehrgpt_output_full"; CEHR_GPT_MODEL_DIR="/home/user/repo/model_run_full"
os.environ.update(OMOP_DIR=OMOP_DIR, CEHR_GPT_DATA_DIR=CEHR_GPT_DATA_DIR, CEHR_GPT_MODEL_DIR=CEHR_GPT_MODEL_DIR)
MODEL_DIR_IN, DATA_DIR = CEHR_GPT_MODEL_DIR, CEHR_GPT_DATA_DIR
DATASET_PREP = f"{DATA_DIR}/dataset_prepared_real"
if REBUILD_PREP and Path(DATASET_PREP).exists(): shutil.rmtree(DATASET_PREP)
for label, pth in [("model/tokenizer dir", MODEL_DIR_IN), ("patient_sequence", f"{DATA_DIR}/patient_sequence")]:
    if not Path(pth).exists(): raise FileNotFoundError(f"[path check] {label} not found: {pth}")

HAS_GPU = torch.cuda.is_available()
print(f"GPU available: {HAS_GPU}" + (f"  ({torch.cuda.get_device_name(0)})" if HAS_GPU else "  -- CPU"))
precision = ["--bf16","true","--fp16","false"] if (HAS_GPU and torch.cuda.is_bf16_supported()) else (["--bf16","false","--fp16","true"] if HAS_GPU else ["--bf16","false","--fp16","false"])
MAX_TOK = str(GPU_MAX_TOKENS) if HAS_GPU else str(MODEL_CTX_LEN)

env = os.environ.copy()
env["OMP_NUM_THREADS"]="8" if HAS_GPU else "16"
env["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"; env["TOKENIZERS_PARALLELISM"]="false"
env["WANDB_DISABLED"]="true"; env["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"]="1"
if not HAS_GPU: env["CUDA_VISIBLE_DEVICES"]=""
env["LD_LIBRARY_PATH"]=f"{sys.prefix}/lib:"+env.get("LD_LIBRARY_PATH","")
env["PYTHONPATH"]=":".join(["/home/user/repo/cehrbert-main","/home/user/repo/cehrgpt-main/src","/home/user/repo/cehrbert_data-main/src",env.get("PYTHONPATH","")])

OUTPUT_DIR = MODEL_DIR_IN.rstrip("/") + "_real"
if FRESH_OUTPUT and Path(OUTPUT_DIR).exists(): shutil.rmtree(OUTPUT_DIR)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True); print("output dir:", OUTPUT_DIR)

cmd = [sys.executable,"-u","-m","cehrgpt.runners.hf_cehrgpt_pretrain_runner",
   "--model_name_or_path",MODEL_DIR_IN,"--tokenizer_name_or_path",MODEL_DIR_IN,"--output_dir",OUTPUT_DIR,
   "--data_folder",f"{DATA_DIR}/patient_sequence","--dataset_prepared_path",DATASET_PREP,"--preprocessing_num_workers","8",
   "--do_train","true","--seed","42","--hidden_size","768","--num_hidden_layers","12",
   "--max_position_embeddings",str(MODEL_CTX_LEN),"--sample_packing","--max_tokens_per_batch",MAX_TOK,
   "--include_values","true",
   "--num_train_epochs",str(NUM_EPOCHS),"--gradient_accumulation_steps","1",
   "--learning_rate","0.0002","--warmup_ratio","0.02","--weight_decay","0.01","--lr_scheduler_type","cosine",
   "--save_strategy","steps","--save_steps",str(SAVE_STEPS),"--save_total_limit","2",
   "--evaluation_strategy","no","--load_best_model_at_end","false","--use_early_stopping","false",
   "--logging_steps","20","--report_to","none","--exclude_position_ids","false",
   "--dataloader_num_workers","4" if HAS_GPU else "0", *precision]
if RESUME:
    cks = sorted(Path(OUTPUT_DIR).glob("checkpoint-*"), key=lambda p:int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else -1)
    if cks: cmd += ["--resume_from_checkpoint", str(cks[-1])]; print("resuming from", cks[-1])

LOGDIR=Path("/home/user/repo/logs"); LOGDIR.mkdir(exist_ok=True)
log_path=LOGDIR/f"train_full-{time.strftime('%Y%m%d-%H%M%S')}.log"
KEEP=re.compile(r'error|exception|traceback|loss|epoch|train_runtime|steps_per_second|saving|checkpoint', re.I)
print(f"\nbatch {MAX_TOK} tok | epochs {NUM_EPOCHS} | save every {SAVE_STEPS}\nlog: {log_path}\n")
n=0; t0=time.time()
with open(log_path,"w") as lf:
    p=subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout:
        lf.write(line); n+=1
        if KEEP.search(line): print(line, end="")
        elif n%500==0: print(f"  ... {n} lines, {(time.time()-t0)/60:.1f} min", flush=True)
rc=p.wait(); print(f"\nrc={rc}  ({(time.time()-t0)/60:.1f} min, log -> {log_path})")
if rc!=0: print("".join(open(log_path).readlines()[-30:]))

## §8 Generate synthetic sequences
Start small (100) to confirm values appear, then scale up.

In [ ]:
import os, sys, re, time, subprocess, torch
from pathlib import Path

# paths — self-initialized, and the model/tokenizer SPLIT made explicit
MODEL_FOLDER     = "/home/user/repo/model_run_full_real"   # trained model: config.json + model.safetensors
TOKENIZER_FOLDER = "/home/user/repo/model_run_full"        # tokenizer: cehrgpt_tokenizer.json + stats
DATA_DIR         = "/home/user/repo/cehrgpt_output_full"   # demographic seeds from real patient_sequence
SYNTH_OUT = Path(globals().get("SYNTH_OUT","/home/user/repo/synthetic_full_v3")); SYNTH_OUT.mkdir(parents=True, exist_ok=True)

# fail fast if the split is wrong (instead of a cryptic load error)
for label, pth, need in [("model", MODEL_FOLDER, "config.json"),
                         ("tokenizer", TOKENIZER_FOLDER, "cehrgpt_tokenizer.json"),
                         ("demographics", f"{DATA_DIR}/patient_sequence", None)]:
    p = Path(pth)
    if not p.exists() or (need and not (p/need).exists()):
        raise FileNotFoundError(f"[path check] {label} wrong: {pth}" + (f" (missing {need})" if need else ""))

HAS_GPU = torch.cuda.is_available()
print(f"GPU: {HAS_GPU}" + (f"  ({torch.cuda.get_device_name(0)})" if HAS_GPU else "  -- CPU (slow!)"))
env = os.environ.copy()
if not HAS_GPU: env["CUDA_VISIBLE_DEVICES"] = ""            # only blank on CPU; otherwise USE the A100
env["OMP_NUM_THREADS"]="8"; env["TOKENIZERS_PARALLELISM"]="false"; env["WANDB_DISABLED"]="true"
env["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"]="1"; env["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"
env["LD_LIBRARY_PATH"] = f"{sys.prefix}/lib:" + env.get("LD_LIBRARY_PATH","")
env["PYTHONPATH"] = ":".join(["/home/user/repo/cehrbert-main",
                              "/home/user/repo/cehrgpt-main/src",
                              "/home/user/repo/cehrbert_data-main/src", env.get("PYTHONPATH","")])

cmd = [sys.executable, "-u", "-m", "cehrgpt.generation.generate_batch_hf_gpt_sequence",
   "--model_folder", MODEL_FOLDER, "--tokenizer_folder", TOKENIZER_FOLDER, "--output_folder", str(SYNTH_OUT),
   "--num_of_patients", "100", "--batch_size", "4", "--buffer_size", "16", "--context_window", "1024",
   "--sampling_strategy", "TopPStrategy", "--top_p", "0.95", "--temperature", "0.9",
   "--repetition_penalty", "1.05", "--epsilon_cutoff", "0.00", "--demographic_data_path", f"{DATA_DIR}/patient_sequence"]

LOGDIR = Path("/home/user/repo/logs"); LOGDIR.mkdir(exist_ok=True)
log_path = LOGDIR / f"generate-{time.strftime('%Y%m%d-%H%M%S')}.log"
KEEP = re.compile(r'error|exception|traceback|generat|batch|patient|saving|wrote', re.I)
print(f"Generating 100 synthetic patients\nlogfile: {log_path}\n")
n=0; t0=time.time()
with open(log_path,"w") as lf:
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout:
        lf.write(line); n+=1
        if KEEP.search(line): print(line, end="")
        elif n % 200 == 0: print(f"  ... {n} lines, {(time.time()-t0)/60:.1f} min", flush=True)
rc=p.wait()
print(f"\nrc={rc}  ({(time.time()-t0)/60:.1f} min, log -> {log_path})")
if rc!=0: print("".join(open(log_path).readlines()[-30:]))

In [ ]:
# CONFIRM the model now generates values natively (this is the v3 acceptance test)
import glob, pyarrow.parquet as pq
gs=sorted(glob.glob(f"{SYNTH_OUT}/**/generated_sequences/*.parquet",recursive=True))
assert gs, "no generated sequences yet"
tot=withval=nval=withcat=0
for f in gs:
    t=pq.read_table(f); cols=t.schema.names
    m=t["concept_value_masks"].to_pylist() if "concept_value_masks" in cols else []
    nv=t["number_as_values"].to_pylist() if "number_as_values" in cols else []
    cv=t["concept_as_values"].to_pylist() if "concept_as_values" in cols else []
    tot+=t.num_rows
    for i in range(t.num_rows):
        if m and m[i] and sum(int(x or 0) for x in m[i])>0: withval+=1
        if nv and nv[i]: nval+=sum(1 for x in nv[i] if x not in (None,0,0.0))
        if cv and cv[i] and any(x not in (None,"") for x in cv[i]): withcat+=1
print(f"{tot:,} synthetic patients | with numeric values {withval:,} ({100*withval/max(tot,1):.0f}%) "
      f"| categorical {withcat:,} | total values {nval:,}")
print("--> non-zero here means NO imputation is needed downstream.")

## §9 Convert to OMOP

In [ ]:
from foundry.transforms import Dataset
import os, shutil, glob
from pathlib import Path

OMOP_DIR = "/home/user/repo/omop_synthea"
out = Path(OMOP_DIR) / "concept"
out.mkdir(parents=True, exist_ok=True)
for f in out.glob("*.parquet"): f.unlink()          # clear any stale copy

downloaded = Dataset.get("copd_synth_sequences").files().download()
srcs = [Path(p) for p in downloaded.values()]
parquet = [p for p in srcs if p.suffix == ".parquet"] or [p for p in srcs if p.is_file()]
if not parquet:
    print("download returned no files:", [p.name for p in srcs][:10])
n = 0
for src in parquet:
    shutil.copy(src, out / src.name); n += 1

print(f"restored {n} concept file(s), "
      f"{len(glob.glob(f'{out}/*.parquet'))} parquet now present -> {out}")

In [ ]:
import os, sys, glob, shutil, re, time, subprocess
from pathlib import Path

# self-init paths (kernel restart clears os.environ)
OMOP_DIR   = "/home/user/repo/omop_synthea"
SYNTH_ROOT = str(SYNTH_OUT)          # matches §8

# locate the generated sequences from Step 3
gen = glob.glob(f"{SYNTH_ROOT}/*/generated_sequences")
assert gen, f"no generated_sequences under {SYNTH_ROOT} — run the generation cell first"
GENERATED = gen[0]
RESTORED  = os.path.join(os.path.dirname(GENERATED), "restored_omop")   # sibling of generated_sequences
shutil.rmtree(RESTORED, ignore_errors=True); os.makedirs(RESTORED, exist_ok=True)

# the converter needs the OMOP concept vocab to route concept_ids to the right domain tables
CONCEPT = f"{OMOP_DIR}/concept"
n_concept = len(glob.glob(CONCEPT + "/*.parquet"))
if n_concept == 0:
    hits = sorted({os.path.dirname(h) for h in glob.glob("/home/user/repo/**/concept/*.parquet", recursive=True)})
    raise FileNotFoundError(
        f"concept vocab not found at {CONCEPT}.\n" +
        ("  found concept parquet here instead — point CONCEPT at one of these:\n    " + "\n    ".join(hits)
         if hits else "  no 'concept' parquet anywhere under repo — re-stage the concept table from Foundry first."))
print(f"generated: {GENERATED}  ({len(glob.glob(GENERATED+'/*.parquet'))} patient files)")
print(f"concept  : {CONCEPT}  ({n_concept} files)")

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = f"{sys.prefix}/lib:" + env.get("LD_LIBRARY_PATH","")
env["PYTHONPATH"] = ":".join(["/home/user/repo/cehrbert-main","/home/user/repo/cehrgpt-main/src",
                              "/home/user/repo/cehrbert_data-main/src", env.get("PYTHONPATH","")])

cmd = [sys.executable, "-u", "-m", "cehrgpt.generation.omop_converter_batch",
   "--patient_sequence_path", GENERATED, "--output_folder", RESTORED, "--concept_path", CONCEPT,
   "--cpu_cores", "8", "--buffer_size", "128"]

LOGDIR = Path("/home/user/repo/logs"); LOGDIR.mkdir(exist_ok=True)
log_path = LOGDIR / f"omop_convert-{time.strftime('%Y%m%d-%H%M%S')}.log"
KEEP = re.compile(r'error|exception|traceback|restored|wrote|convert|table|person|visit|condition', re.I)
print(f"logfile: {log_path}\n")
n=0; t0=time.time()
with open(log_path,"w") as lf:
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout:
        lf.write(line); n+=1
        if KEEP.search(line): print(line, end="")
        elif n % 200 == 0: print(f"  ... {n} lines, {(time.time()-t0)/60:.1f} min", flush=True)
rc=p.wait()
print(f"\nrc={rc}  ({(time.time()-t0)/60:.1f} min, log -> {log_path})")
if rc==0:
    print("\n--- restored OMOP tables ---")
    tbls = [d for d in sorted(Path(RESTORED).iterdir()) if d.is_dir()]
    if not tbls: print("  (no tables written — check the log)")
    for d in tbls:
        ps = list(d.glob("*.parquet"))
        print(f"  {d.name:28s} {len(ps):3d} files  {sum(pf.stat().st_size for pf in ps)/1024:8.1f} KB")
else:
    print("".join(open(log_path).readlines()[-30:]))

### §9b Post-process: birthdates + timeline jitter

Writes `restored_omop_pp` alongside `restored_omop`; the raw conversion is left untouched.

In [ ]:
# ---- compute per-person birth info + first-visit-anchored time shift ----
import glob, numpy as np, pandas as pd
RESTORED=glob.glob(f"{SYNTH_OUT}/*/restored_omop")[0]
print("post-processing:", RESTORED)

person=pd.concat([pd.read_parquet(f) for f in glob.glob(f"{RESTORED}/person/*.parquet")], ignore_index=True)
BYC="year_of_birth" if "year_of_birth" in person.columns else "birth_year"
yob=person.groupby("person_id")[BYC].first()

firsts=[]
for f in glob.glob(f"{RESTORED}/visit_occurrence/*.parquet"):
    v=pd.read_parquet(f, columns=["person_id","visit_start_date"])
    firsts.append(v.groupby("person_id")["visit_start_date"].min())
first=pd.to_datetime(pd.concat(firsts).groupby(level=0).min())

fy=first.dt.year
ystart=pd.to_datetime(dict(year=fy,month=1,day=1))
yend  =pd.to_datetime(dict(year=fy,month=12,day=31))
byr=yob.reindex(first.index); byr=byr.where(byr.notna(),fy).astype(int)
bstart=pd.to_datetime(dict(year=byr,month=1,day=1))
lo=np.maximum(ystart.values,bstart.values)                # stay in-year AND after birth
span=((yend.values-lo)/np.timedelta64(1,"D")).clip(min=0).astype(int)
rnd=np.floor(np.random.random(len(first))*(span+1)).astype(int)
newf=pd.to_datetime(lo)+pd.to_timedelta(rnd,unit="D")
shift=pd.Series(((newf.values-first.values)/np.timedelta64(1,"D")).astype("int64"),
                index=first.index, name="shift_days")
print(f"time-shift computed for {len(shift):,} persons | median |shift| {int(np.median(np.abs(shift)))} days")

In [ ]:
# ---- apply the shift to EVERY event table + randomize birthdates (streaming) ----
import os, glob, numpy as np, pandas as pd
OUT=os.path.join(os.path.dirname(RESTORED),"restored_omop_pp"); os.makedirs(OUT, exist_ok=True)

for tbl in ["visit_occurrence","condition_occurrence","drug_exposure",
            "measurement","procedure_occurrence","death","observation"]:
    files=glob.glob(f"{RESTORED}/{tbl}/*.parquet")
    if not files: continue
    dst=f"{OUT}/{tbl}"; os.makedirs(dst, exist_ok=True); dcols=None
    for f in files:
        df=pd.read_parquet(f)
        if dcols is None: dcols=[c for c in df.columns if "date" in c.lower()]
        td=pd.to_timedelta(df["person_id"].map(shift).fillna(0).astype("int64"), unit="D")
        for c in dcols:
            s=pd.to_datetime(df[c], errors="coerce")+td
            df[c]=s.dt.date if c.lower().endswith("_date") else s
        df.to_parquet(f"{dst}/{os.path.basename(f)}", index=False)
    print(f"  shifted {tbl:24s} {len(files):5d} files  cols={dcols}")

pdst=f"{OUT}/person"; os.makedirs(pdst, exist_ok=True)
for f in glob.glob(f"{RESTORED}/person/*.parquet"):
    df=pd.read_parquet(f); n=len(df)
    mob=np.random.randint(1,13,n)
    dim=pd.to_datetime(dict(year=df[BYC],month=mob,day=1)).dt.daysinmonth
    dob=(np.floor(np.random.random(n)*dim).astype(int)+1)
    if "month_of_birth" in df.columns: df["month_of_birth"]=mob
    if "day_of_birth"   in df.columns: df["day_of_birth"]=dob
    df["birth_datetime"]=pd.to_datetime(dict(year=df[BYC],month=mob,day=dob))
    df.to_parquet(f"{pdst}/{os.path.basename(f)}", index=False)
print("  randomized person birthdates")
print("\n->", OUT)

In [ ]:
# ---- verify the post-processed output ----
import glob, os, pyarrow.parquet as pq
PP=glob.glob(f"{SYNTH_OUT}/*/restored_omop_pp")[0]
for t in sorted(d for d in os.listdir(PP) if os.path.isdir(f"{PP}/{d}")):
    fs=glob.glob(f"{PP}/{t}/*.parquet")
    raw=glob.glob(f"{RESTORED}/{t}/*.parquet")
    r_pp=sum(pq.read_metadata(x).num_rows for x in fs)
    r_raw=sum(pq.read_metadata(x).num_rows for x in raw) if raw else 0
    print(f"  {t:24s} {r_pp:>12,} rows  {'OK' if r_pp==r_raw else f'!= raw {r_raw:,}'}")
p=pq.read_table(glob.glob(f"{PP}/person/*.parquet")[0]).slice(0,2).to_pylist()
print("\nperson sample:", [{k:v for k,v in r.items() if k in ("person_id","year_of_birth","birth_datetime")} for r in p])
v=pq.read_table(glob.glob(f"{PP}/visit_occurrence/*.parquet")[0], columns=["visit_start_date"]).slice(0,3).to_pylist()
print("visit_start_date sample:", v)

## §10 Persist the trained model

In [ ]:
# Persist the FINISHED full model + tokenizer to Foundry (run AFTER training completes).
# Bundles model_run_full_real (final weights; intermediate checkpoints excluded) + model_run_full (tokenizer).
import subprocess
from foundry.transforms import Dataset

TARGET = "cehrgpt_model_full"
r = subprocess.run(["bash","-lc", r"""
set -e
STAGE=/home/user/repo/_bundle_full
rm -rf "$STAGE"; mkdir -p "$STAGE/model" "$STAGE/tokenizer"
( cd /home/user/repo/model_run_full_real && tar cf - --exclude='checkpoint-*' . ) | tar xf - -C "$STAGE/model"
cp -a /home/user/repo/model_run_full/. "$STAGE/tokenizer/"
cd "$STAGE" && tar czf /home/user/repo/cehrgpt_full_bundle.tar.gz .
echo "=== bundle top level ==="; tar tzf /home/user/repo/cehrgpt_full_bundle.tar.gz | grep -E '^\./(model|tokenizer)/[^/]+$' | sort
echo "=== size ==="; ls -lh /home/user/repo/cehrgpt_full_bundle.tar.gz
rm -rf "$STAGE"
"""], capture_output=True, text=True)
print(r.stdout[-2500:])
if r.returncode: print("ERR:", r.stderr[-800:])
Dataset.get(TARGET).upload_file("/home/user/repo/cehrgpt_full_bundle.tar.gz", "cehrgpt_full_bundle.tar.gz")
print("✓ full model bundle (model + tokenizer) uploaded to", TARGET)
